In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag=int(dbutils.widgets.get("init_load_flag"))

Data Reading from source



In [0]:
df= spark.sql("select * from databricks_cat.silver.customers_silver")

Removing Dupliucates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])


In [0]:
# df = df.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))

In [0]:
df.display()

## Dividing new vs old


In [0]:
if init_load_flag==0:

    df_old=spark.sql('''select DimCustomerKey, customer_id, create_date, update_date
                     from databricks_cat.gold.DimCustomers''')
    
else:
    df_old= spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
                     from databricks_cat.silver.customers_silver where 1=0''')

In [0]:
df_old.display()

Renaming Columns of df_old

In [0]:
df_old= df_old.withColumnRenamed("DimCustomerKey","old_DimCustomerKey")\
                  .withColumnRenamed("customer_id","old_customer_id")\
                  .withColumnRenamed("create_date","old_create_date")\
                  .withColumnRenamed("update_date","old_update_date")

## Applying the joins

In [0]:
df_join = df.join(df_old, df['customer_id'] == df_old['old_customer_id'], 'left')
#df_join.display()

In [0]:
df_join.display()

Seperating New vs Old

In [0]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())


In [0]:
df_old= df_join.filter(df_join['old_DimCustomerKey'].isNotNull())


## Preaparing df_old

In [0]:
#droppping all the columns which are not required
df_old = df_old.drop('old_customer_id','old_update_date') 

# renaming old dim customer key
df_old = df_old.withColumnRenamed("old_DimCustomerKey","DimCustomerKey")

#renaming creating old_create_date to create date
df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))


#Recreating update_date columns with current timestamp
df_old= df_old.withColumn("update_date", current_timestamp())

In [0]:
df_old.display()


Preparing df_new

In [0]:
#droppping all the columns which are not required
df_new = df_new.drop('old_DimCustomerKey','old_customer_id','old_update_date', 'old_create_date') 

#Recreating update_date, currentdate columns with current timestamp
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())

In [0]:
df_new.display()

SURROGATE KEY FROM 1

In [0]:
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

Adding max surrogate key



In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_cat.gold.DimCustomers")
    
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']



In [0]:
df_new = df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))

Union of df_old and df_new

In [0]:

df_final = df_new.unionByName(df_old)


In [0]:
df_final.display()

Reanaming


### SCD TYPE 1

In [0]:
from delta.tables import DeltaTable

In [0]:
if (spark.catalog.tableExists("databricks_cat.gold.DimCustomers")):
    dlt_obj = DeltaTable.forPath(spark,"abfss://gold@databricksprojectanushaa.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey = src.DimCustomerKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@databricksprojectanushaa.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("databricks_cat.gold.DimCustomers")

In [0]:
%sql
SELECT * FROM databricks_cat.gold.dimcustomers